# Universidad Tecmilenio
## Curso: Desarrollo de Aplicaciones con IA
### Actividad: Avance de Proyecto 1
**Asistente Inteligente para la Inspección Visual y Trazabilidad de Daños en Empaques durante la Recepción de Mercancías**

### Contexto de Negocio y Dominio del Proyecto
- **Propósito:** Asistir a inspectores de almacén en andenes de carga para evaluar paquetes (empaques intactos vs. dañados).
- **Alcance actual (Avance 1):** Chatbot conversacional basado en reglas y expresiones regulares desarrollado en Python con **NLTK** (tokenización y stemming en español), extracción de identificadores operativos (lote y andén), clasificación de intenciones por coincidencia de palabras/frases clave (`keywords`/`frases_clave`), gestión de sesión con llenado de datos (*slot-filling*), emisión de respuestas y fallbacks controlados, y bitácora transaccional.

- **Evolución posterior (Avances 2 y Proyecto Final):** Se integrará un clasificador supervisado con **TF-IDF + SVM** para clasificación de intenciones, una CNN binaria (**ResNet50**) para clasificación visual de empaques (`caja_intacta` vs. `caja_danada`) y una interfaz web en **Streamlit** con API en **Flask**.

### Arquitectura de Componentes (Avance 1 a Proyecto Final)

La solución está estructurada modularmente bajo el principio de inversión de dependencias (`BaseIntentClassifier`), lo cual permite desacoplar el gestor de diálogo de la técnica de clasificación (reglas simbólicas vs. Machine Learning):

```
┌─────────────────────────────────────────────────────────────┐
│                      Inspector (Usuario)                     │
└──────────────────────────────┬──────────────────────────────┘
                               │ Texto natural
                               ▼
┌─────────────────────────────────────────────────────────────┐
│               DialogueManager (Gestor de Diálogo)            │
│  - Enrutamiento conversacional, slots pendientes y fallback  │
└──────────┬───────────────────┬───────────────────┬──────────┘
           │                   │                   │
           ▼                   ▼                   ▼
┌─────────────────────┐ ┌────────────────┐ ┌───────────────────┐
│ BaseIntentClassifier│ │ EntityExtractor│ │  AuditoriaSession │
│ (RuleBased/ML TFIDF)│ │ (Lote, Andén)  │ │ (Estado y Reglas) │
└──────────┬──────────┘ └────────────────┘ └─────────┬─────────┘
           │                                         │
           ▼                                         ▼
┌─────────────────────┐                     ┌───────────────────┐
│  data/intents.json  │                     │    bitacora.json  │
└─────────────────────┘                     └───────────────────┘
```

### Módulo 1: Preprocesamiento de Lenguaje Natural (NLTK)
Aplica técnicas formales de limpieza, normalización NFD (remoción de acentos), tokenización léxica (`wordpunct_tokenize`) y reducción morfológica por stemming para español (`SnowballStemmer`).

In [ ]:
from src.preprocessor import normalizar, preprocesar

ejemplo_texto = "¡Alerta! Las cajas llegaron-dañadas, perforadas y mojadas en el andén-2."
print("Texto original :", ejemplo_texto)
print("Normalizado    :", normalizar(ejemplo_texto))
print("Tokens + Stems :", preprocesar(ejemplo_texto))

Texto original : ¡Alerta! Las cajas llegaron-dañadas, perforadas y mojadas en el andén-2.
Normalizado    : alerta las cajas llegaron-danadas perforadas y mojadas en el anden-2
Tokens + Stems : ['alert', 'las', 'caj', 'lleg', 'dan', 'perfor', 'y', 'moj', 'en', 'el', 'anden', '2']


### Módulo 2: Extracción de Entidades Operativas (RegEx y Slot-Filling)
Extrae de forma robusta identificadores de lote (ej. `LOTE-504`, `710`) y andenes o bahías de recepción (ej. `andén 2`, `bahía 3`), soportando respuestas directas con números aislados para llenado de slots.

In [3]:
from src.entity_extractor import extraer_entidades, extraer_numero_aislado

msg_1 = "Voy a auditar el lote 504 en el andén 2"
msg_2 = "Revisando llegada de refacciones en bahía 3 batch N-90"
msg_3 = "3"

print(f"Mensaje: '{msg_1}' -> Entidades:", extraer_entidades(msg_1))
print(f"Mensaje: '{msg_2}' -> Entidades:", extraer_entidades(msg_2))
print(f"Mensaje: '{msg_3}' -> Número aislado (slot):", extraer_numero_aislado(msg_3))

Mensaje: 'Voy a auditar el lote 504 en el andén 2' -> Entidades: {'lote': '504', 'anden': 2}
Mensaje: 'Revisando llegada de refacciones en bahía 3 batch N-90' -> Entidades: {'lote': 'N-90', 'anden': 3}
Mensaje: '3' -> Número aislado (slot): 3


### Módulo 3: Base de Conocimiento y Motor de Reglas (Chatbot Engine)
Carga el catálogo `data/intents.json` e indexa raíces de palabras y secuencias de frases compuestas. La interfaz `BaseIntentClassifier` garantiza compatibilidad directa con futuros clasificadores de Machine Learning.

In [2]:
from src.chatbot_engine import RuleBasedChatbot

bot = RuleBasedChatbot("data/intents.json")
print("Intenciones cargadas:", bot.obtener_intenciones())

# Pruebas de clasificación directa
frases_test = [
    "Hola, buen día",
    "Voy a auditar el lote 504 en el andén 2",
    "La caja viene rota de una esquina",
    "Las cajas vienen en perfecto estado",
    "¿Cuáles son los motivos para rechazar una caja?",
    "Quiero pedir comida"
]

for f in frases_test:
    tag = bot.clasificar(f)
    resp = bot.obtener_respuesta_base(tag)
    print(f"'{f}' -> [{tag}]")
    print(f"Respuesta base: {resp}\n")

Intenciones cargadas: ['saludo', 'iniciar_auditoria', 'reportar_dano_empaque', 'confirmar_empaque_intacto', 'consultar_politica_rechazo', 'despedida', 'fallback']
'Hola, buen día' -> [saludo]
Respuesta base: ¡Hola! Soy tu asistente virtual para auditoría de almacén. Puedo ayudarte a auditar lotes, registrar daños de empaque y proporcionarte información sobre políticas de rechazo. ¿Cómo deseas proceder?

'Voy a auditar el lote 504 en el andén 2' -> [iniciar_auditoria]
Respuesta base: Auditoría iniciada. Por favor, procede a revisar el lote y confirma si los empaques están en buen estado o si presentan algún daño.

'La caja viene rota de una esquina' -> [reportar_dano_empaque]
Respuesta base: Alerta de daño registrada. De acuerdo con el protocolo de calidad, no introduzcas el paquete al inventario general. Por favor, segrega el bulto en la zona de cuarentena y solicita la firma de rechazo al transportista.

'Las cajas vienen en perfecto estado' -> [confirmar_empaque_intacto]
Respuesta ba

### Módulo 4: Gestor de Estado y Reglas de Negocio (Auditoría Session)
Controla el ciclo de vida de la auditoría, gestiona los identificadores (lote y andén/bahía), maneja la completitud de slots y dictamina el estatus de calidad del empaque (`RECHAZADO_CUARENTENA` vs. `APROBADO_LIBERADO`).

In [4]:
from src.auditoria_manager import AuditoriaSession

sesion = AuditoriaSession(responsable="Inspector_Anden")
sesion.iniciar_inspeccion(lote="LOTE-504", anden=24)
print("¿Identificadores completos? :", sesion.identificadores_completos)
print(f"Lote activo: {sesion.lote} | Andén activo: {sesion.anden}")

dictamen = sesion.evaluar_empaque(es_danado=False)
print("Dictamen emitido :", dictamen)
print("Resumen sesión   :", sesion.obtener_resumen_reporte())

¿Identificadores completos? : True
Lote activo: LOTE-504 | Andén activo: 24
Dictamen emitido : APROBADO_LIBERADO
Resumen sesión   : {'responsable': 'Inspector_Anden', 'fecha': '2026-10-24 10:21:34', 'lote': 'LOTE-504', 'anden': 24, 'dictamen': 'APROBADO_LIBERADO', 'en_progreso': True, 'total_eventos': 1}


### Módulo 5: Orquestador Conversacional (Dialogue Manager)
Integra el clasificador, la extracción de entidades y el gestor de estado para coordinar el diálogo continuo, gestionar slots incompletos y emitir fallbacks contextualizados.

In [1]:
from src.dialogue_manager import DialogueManager
from src.chatbot_engine import RuleBasedChatbot
from src.auditoria_manager import AuditoriaSession

bot = RuleBasedChatbot("data/intents.json")
gestor = DialogueManager(classifier=bot, session=AuditoriaSession())

dialogo_demo = [
    "Buenas tardes, quiero iniciar turno",
    "Revisando llegada de refacciones en bahía 3",
    "lote 710",
    "La caja viene rota de una esquina",
    "Es todo por el momento"
]

for entrada in dialogo_demo:
    resp = gestor.responder(entrada)
    print(f"Inspector : {entrada}")
    print(f"Bot       : {resp}")
    print("-" * 60)

Inspector : Buenas tardes, quiero iniciar turno
Bot       : ¡Hola! Soy tu asistente virtual para auditoría de almacén. Puedo ayudarte a auditar lotes, registrar daños de empaque y proporcionarte información sobre políticas de rechazo. ¿Cómo deseas proceder?
------------------------------------------------------------
Inspector : Revisando llegada de refacciones en bahía 3
Bot       : Entendido. ¿Cuál es el número de lote que vas a auditar?
------------------------------------------------------------
Inspector : lote 710
Bot       : Entendido. He registrado el inicio de auditoría del lote 710 en el andén 3. Por favor, realiza la inspección física del paquete y describe si el empaque presenta golpes o si se encuentra intacto.
------------------------------------------------------------
Inspector : La caja viene rota de una esquina
Bot       : [Lote 710 · andén 3] Alerta de daño registrada. De acuerdo con el protocolo de calidad, no introduzcas el paquete al inventario general. Por favor,

Módulo 6: Batería de Pruebas Automatizadas (Dataset JSON y Reportes en DataFrame)
Ejecución de las 10 conversaciones oficiales (43 turnos en total) cargadas desde `data/test_conversaciones.json`. Evalúa la precisión de intenciones, tasa de fallback, genera reportes estructurados en DataFrames de pandas y exporta la bitácora transaccional a `data/bitacora_pruebas.json`.

In [1]:
from tests.run_tests import ejecutar_evaluacion

# Ejecuta la batería de pruebas cargando data/test_conversaciones.json
resultados = ejecutar_evaluacion(verbose=True)


 BATERÍA DE PRUEBAS FUNCIONALES - ASISTENTE DE AUDITORÍA DE ALMACÉN
 Dataset de evaluación cargado desde: data/test_conversaciones.json

--- C1 Flujo completo conforme ---
Usuario : Hola, buen día
Bot     : Hola. Soy el Asistente de Auditoría de Almacén. Puedo ayudarte a registrar inspecciones de carga, consultar criterios de empaque y generar reportes de no-conformidad.
          [Esperada: saludo                     | Detectada: saludo                    ] [OK]
Usuario : Voy a auditar el lote 504 en el andén 2
Bot     : Entendido. He registrado el inicio de auditoría del lote 504 en el andén 2. Por favor, realiza la inspección física del paquete y describe si el empaque presenta golpes o si se encuentra intacto.
          [Esperada: iniciar_auditoria          | Detectada: iniciar_auditoria         ] [OK]
Usuario : Las cajas vienen en perfecto estado
Bot     : [Lote 504 · andén 2] Excelente. Inspección de empaque aprobada. El material ha sido validado para su liberación e ingreso al 

In [4]:
# Visualiza los DataFrames con el formato de tabla interactiva de Jupyter:
print("\n--- REPORTE CONSOLIDADO DE MÉTRICAS ---")
display(resultados["df_metricas"])

print("\n--- DETALLE POR CONVERSACIÓN ---")
display(resultados["df_detalle"])

print("\n--- EVALUACIÓN DE PRUEBAS DE LÍMITES ---")
display(resultados["df_limites"])


--- REPORTE CONSOLIDADO DE MÉTRICAS ---


,Métrica,Valor
0,Conversaciones evaluadas,10
1,Conversaciones sin error,10/10 (100.0%)
2,Total de turnos evaluados,43
3,Turnos con intención correcta,43/43
4,Precisión global (Accuracy),100.0%
5,Tasa de Fallback controlado,7.0%
6,Eventos registrados en bitácora,43



--- DETALLE POR CONVERSACIÓN ---


,ID,Conversación,Turnos,Aciertos,Precisión,Estatus
0,C1,C1 Flujo completo conforme,4,4,100.0%,APROBADA
1,C2,C2 Daño con datos incompletos,5,5,100.0%,APROBADA
2,C3,C3 Consulta de políticas,4,4,100.0%,APROBADA
3,C4,C4 Actualización de andén y confirmación,4,4,100.0%,APROBADA
4,C5,C5 Daño sin auditoría activa,4,4,100.0%,APROBADA
5,C6,C6 Mensajes fuera de dominio,6,6,100.0%,APROBADA
6,C7,C7 Datos por separado,4,4,100.0%,APROBADA
7,C8,C8 Respuesta solo con número,3,3,100.0%,APROBADA
8,C9,C9 Política antes de auditar,4,4,100.0%,APROBADA
9,C10,C10 Daño y consulta de criterios,5,5,100.0%,APROBADA



--- EVALUACIÓN DE PRUEBAS DE LÍMITES ---


,Caso de Límite,Mensaje Inspector,Deseada,Detectada,Resultado
0,Errata en palabra clave,la caja esta aplastda,reportar_dano_empaque,fallback,Limitación Reglas Léxicas
1,Sinónimo fuera del diccionario,la caja trae un hueco enorme,reportar_dano_empaque,fallback,Limitación Reglas Léxicas
2,Sinónimo no contemplado,el bulto viene en mal estado,reportar_dano_empaque,fallback,Limitación Reglas Léxicas
3,Mensaje ambiguo,hay una caja pero no sé si está bien,fallback,fallback,Superada
